In [1]:
# =========================
# 1. LIBRARIES
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

In [ ]:
# =========================
# 2. DATA GENERATION (SIMULATED TIDAL DATA)
# =========================
base_path = Path(r"C:\Users\Kunde\Documents\Project-zum-Lebenslauf\GitHub_Projects\GIS-Tidal-Zones-Analysis")
data_path = base_path / "data"
data_path.mkdir(parents=True, exist_ok=True)

years = [1982, 1992, 2002, 2012, 2018]
basins = [f"Basin_{i}" for i in range(1, 9)]
zones = ["Subtidal", "Intertidal", "Supratidal"]

np.random.seed(42)

data = []

for b in basins:
    for y in years:
        for z in zones:
            data.append([
                b,
                y,
                z,
                np.random.uniform(50, 200),   # Area
                np.random.uniform(-2, 3)     # Height
            ])

df = pd.DataFrame(data, columns=["Basin", "Year", "Zone", "Area", "Height"])

df.to_csv(data_path / "tidal_data.csv", index=False)
print(df.head())

In [ ]:
# =========================
# 3. STATISTICAL ANALYSIS
# =========================

# Zone + Year Trend
zone_trend = df.groupby(["Year", "Zone"])["Area"].mean().reset_index()

# Individual Zone Trends
def get_trend(zone):
    return df[df["Zone"] == zone].groupby("Year")["Area"].mean()

df

In [ ]:
# =========================
# 4. PLOT (CLEAN SCIENTIFIC STYLE)
# =========================

colors = {
    "Subtidal": "blue",
    "Intertidal": "yellow",
    "Supratidal": "red"
}

plt.figure(figsize=(8,5))

for z in zones:
    trend = get_trend(z)
    plt.plot(trend.index, trend.values, marker="o", label=z, color=colors[z])

plt.title("Coastal Zone Area Change Over Time")
plt.xlabel("Year")
plt.ylabel("Mean Area (m²)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =========================
# 5. DATABASE (SQLite)
# =========================

from pathlib import Path
import sqlite3
import pandas as pd

base_path = Path(r"C:\Users\Kunde\Documents\Project-zum-Lebenslauf\GitHub_Projects\GIS-Tidal-Zones-Analysis")
data_path = base_path / "data"

db_path = data_path / "tidal.db"

conn = sqlite3.connect(db_path)

# remove geometry (IMPORTANT)
clean_df = gdf.drop(columns="geometry")

# USE CLEAN DF HERE (not df)
clean_df.to_sql("tidal_data", conn, if_exists="replace", index=False)

query = """
SELECT Year, Zone, AVG(Area) AS avg_area
FROM tidal_data
GROUP BY Year, Zone
"""

result = pd.read_sql(query, conn)
print(result.head())

conn.close()

In [ ]:
# =========================
# 6. GIS (GeoDataFrame)
# =========================

coords = {
    "Subtidal": (2, 2),
    "Intertidal": (1, 1),
    "Supratidal": (3, 3)
}

df["geometry"] = df["Zone"].map(lambda z: Point(coords[z]))

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

print(gdf.head())

In [ ]:
# =========================
# 7. EXPORT GIS FILES
# =========================

out_gpkg = data_path / "tidal_data.gpkg"

gdf.to_file(out_gpkg, layer="tidal_zones", driver="GPKG")
print("Saved:", out_gpkg)

In [ ]:
# =========================
# 8. EXPORT GIS FILES
# =========================
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

gdf[gdf["Zone"] == "Subtidal"].plot(ax=ax, color="blue", label="Subtidal")
gdf[gdf["Zone"] == "Intertidal"].plot(ax=ax, color="yellow", label="Intertidal")
gdf[gdf["Zone"] == "Supratidal"].plot(ax=ax, color="red", label="Supratidal")

plt.title("Tidal Zones GIS Visualization")

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid(True)

plt.show()

#### Einblick:  8. EXPORT GIS FILES 
```
- Subtidal → Höhere und stabilere Wasserstände → Stabile Umgebung 
- Intertidal → Hohe Schwankungen → Übergangszone 
- Supratidal → Hohe Dispersion + negative Werte → Reagiert empfindlich auf Umweltbedingungen
```

---- 

## Seaborn Plots

In [ ]:
# =========================
# 1. Boxplot: Höhe der Zone
# =========================

import seaborn as sns
import matplotlib.pyplot as plt

# custom colors
palette = {
    "Subtidal": "blue",
    "Intertidal": "yellow",
    "Supratidal": "red"
}

plt.figure(figsize=(8,5))

sns.boxplot(
    data=gdf,
    x="Zone",
    y="Height",
    palette=palette
)

plt.title("Height Distribution Across Tidal Zones")
plt.xlabel("Zone")
plt.ylabel("Height")

plt.show()

####  Einblick:  1. Höhe der Zone:
```
- Subtidal → Höhere und stabilere Wasserstände → Stabile Umgebung ✔️
- Intertidal → Hohe Schwankungen → Übergangszone ⚠️
- Supratidal → Hohe Dispersion + negative Werte → Empfindlich gegenüber Umweltbedingungen ⚠️
```

In [ ]:
# ========================= 
# 2. Violinendiagramm (Reelle Verteilung)
# =========================

import seaborn as sns
import matplotlib.pyplot as plt


import seaborn as sns
import matplotlib.pyplot as plt

# custom colors
palette = {
    "Subtidal": "blue",
    "Intertidal": "yellow",
    "Supratidal": "red"
}

plt.figure(figsize=(8,5))

sns.violinplot(
    data=gdf,
    x="Zone",
    y="Height",
    inner="quartile",
    palette=palette
)

plt.title("Height Distribution Shape per Zone")
plt.xlabel("Zone")
plt.ylabel("Height")

plt.show()


In [ ]:
# =========================
# 3. Trend im Laufe der Zeit
# =========================

import seaborn as sns
import matplotlib.pyplot as plt

# custom colors
palette = {
    "Subtidal": "blue",
    "Intertidal": "yellow",
    "Supratidal": "red"
}

plt.figure(figsize=(10,5))

sns.lineplot(
    data=gdf,
    x="Year",
    y="Height",
    hue="Zone",
    palette=palette,
    marker="o"
)

plt.title("Temporal Change of Height in Tidal Zones")
plt.xlabel("Year")
plt.ylabel("Height")

plt.show()

### Einblick: 3. Zeitliche Einsicht:
```
- Abnahme des Subtidalbereichs → mögliche Erosion oder Meeresspiegelanstieg
- Instabilität des Intertidalbereichs → Klimastress / Sedimentverlagerung
- Zunahme/Abnahme des Supratidalbereichs → Einfluss extremer Wetterereignisse
```

In [ ]:
# =========================
# 4. Beckenvergleich
# =========================

import seaborn as sns
import matplotlib.pyplot as plt

# custom colors
palette = {
    "Subtidal": "blue",
    "Intertidal": "yellow",
    "Supratidal": "red"
}

plt.figure(figsize=(10,5))

sns.barplot(
    data=gdf,
    x="Basin",
    y="Height",
    hue="Zone",
    palette=palette
)

plt.title("Height Comparison Across Basins")
plt.xlabel("Basin")
plt.ylabel("Mean Height")

plt.xticks(rotation=45)

plt.show()




### Einblick Vergleichende Analyse: 4. Beckenvergleich 
```
1. Subtidal
- Positiver und stabiler Wasserstand
- Geringste jährliche Veränderungen
✔️ Guter Zustand (stabile Meereszone)

2. Intertidal
- Höchste Variabilität
- Sensitiv gegenüber Jahr und Becken
⚠️ Zone mit mäßiger/überschreitender Belastung

3. Supratidal
- Beinhaltet negative Werte
- Hinweis auf Austrocknung oder unregelmäßige Sedimentation
⚠️ In einigen Becken ökologisch instabil
```